# XR Client Visuals

Examples of dynamically creating simple visuals in the XR client from python code.

## Setup runner & utilities

In [ ]:
from nanover.app import OmniRunner
from nanover.openmm import OpenMMSimulation

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="EXAMPLE: xr client visuals")
imd_runner.load(0)

In [ ]:
from nanover.jupyter import NanoverJupyterUtilities

utilities = NanoverJupyterUtilities.from_runner(imd_runner)

## Kinetic energy graph

In [ ]:
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener


class EnergyGraph(FrameListener):
    _energies = [0]

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        limit = 30 * 60
        self._energies = self._energies[-limit+1:] + [full_frame.kinetic_energy]

        y_min = 0
        y_max = max(self._energies)

        averages = [sum(self._energies[i:i+10])/len(self._energies[i:i+10]) for i in range(0, len(self._energies), 10)]

        Y = [(y - y_min) / (y_max - y_min) for y in averages]
        points = [(i/len(averages) * 10, y * 10, 0.0) for i, y in enumerate(Y)]
        window = points[-8:]
        label_y = sum(y for _, y, _ in window) / len(window)
        label_pos = (window[-1][0], label_y, 0.0)

        with utilities.objects as objects:
            objects.update_line("graph.energy.kinetic", positions=points, size=0.02, color=[1.0, 1.0, 1.0, .5])
            objects.update_label("graph.energy.kinetic", position=label_pos, color=[1.0, .5, .5, 1.0], text=f"kinetic energy:\n{self._energies[-1]:.2f}")


energy_graph = EnergyGraph.from_runner(imd_runner)
energy_graph.start()

## Length tracking display

In [ ]:
import numpy as np
from nanover.trajectory import FrameData
from nanover.jupyter import FrameListener


class LengthVisuals(FrameListener):
    _pairs: list[tuple[int, int]] = []

    def clear(self):
        utilities.objects.clear()
        self._pairs.clear()

    def add_pair(self, atom_a: int, atom_b: int):
        self._pairs.append((atom_a, atom_b))

    def on_frame_update(self, full_frame: FrameData, frame_update: FrameData):
        with utilities.objects as objects:
            for i, (atom_a, atom_b) in enumerate(self._pairs):
                pos_a = full_frame.particle_positions[atom_a]
                pos_b = full_frame.particle_positions[atom_b]
                length = np.linalg.norm(pos_a - pos_b)
                centroid = (pos_a + pos_b) / 2

                objects.update_line(i, positions=(pos_a, pos_b), size=0.01, color=[.5, .5, .5, .5])
                objects.update_label(i, position=centroid, text=f"{length:.2f}nm")

length_visuals = LengthVisuals.from_runner(imd_runner)
length_visuals.start()

imd_runner.app_server.register_command("user/lengths/clear", length_visuals.clear)

In [ ]:
from nanover.imd import ParticleInteraction
from nanover.jupyter import Mode


class LengthMode(Mode):
    prev_atom = None

    def on_interaction_started(self, *, key: str, interaction: ParticleInteraction):
        next_atom = int(interaction.particles[0])

        if self.prev_atom is None:
            self.prev_atom = next_atom
            utilities.notify_all(f"pick next atom")
        else:
            length_visuals.add_pair(next_atom, self.prev_atom)
            utilities.notify_all(f"tracking length")
            self.prev_atom = None


utilities.use_interaction_modes()
utilities.add_interaction_mode(LengthMode, "length")